# 浏览器：动作之后为什么还要观察

本章学习目标：

- 区分 API 返回、HTML、DOM、可见正文和截图。
- 用 Playwright 读取页面、定位按钮和点击。
- 处理弹窗、加载等待和定位失败。
- 限定浏览器可以访问和执行的操作。
- 保存动作前后的页面，判断操作是否产生预期结果。

本页先用本地课堂页面检查浏览器流程。下一页再解释真实模型读取公开网页的实验。

[返回课程入口](../README.md) · 理解基本循环后再看。


## 1. 浏览器提供哪些观察

| 表示 | 内容 | 需要注意什么 |
| --- | --- | --- |
| 初始 HTML | 页面最初的源代码 | 不等于脚本执行后的界面 |
| 当前 DOM | 当前页面的元素、属性和文本 | 可能包含隐藏元素 |
| 可见正文 | 用户此时能读到的文本 | 可能丢失布局和图片信息 |
| 截图 | 页面像素、布局和遮挡 | 需要视觉理解，不能直接当作文本输入 |

本课程文本模型接收可见正文，截图用于复盘，没有送入视觉模型。保存截图本身不能证明模型看懂了画面。


## 2. 看一个“读取 → 点击 → 再读取”的任务

本地页面有“显示证据”按钮，初始隐藏一句话。点击后先出现演示提示框，再延迟显示：

> Agent 根据工具反馈选择下一步行动。

目标是取得这句实际可见的正文。合理流程是：

1. 打开页面，读取当前正文和按钮。
2. 点击“显示证据”。
3. 按本任务规则处理演示提示框。
4. 等待证据元素可见。
5. 再次读取页面，核对新增文本。

点击成功只说明动作执行过，不能代替点击后的观察。


## 3. 定位失败、等待和重试有什么区别

默认程序先尝试“旧按钮名称”，预期会超时；随后重新保存页面，按当前角色和名称定位“显示证据”。这条恢复顺序写在代码里，是固定工作流，不是模型自行发现了按钮变化。

- **等待**：在期限内等目标条件出现，例如证据变为可见。
- **重试**：再次执行某个操作。
- **重新规划**：读到新情况后，选择另一个操作。

按钮改名时，盲目等得更久可能没用；内容延迟出现时，立即重试点击也可能多余。定位时使用当前页面的角色、名称等信息，仍要处理重名或属性缺失。


## 4. 为什么有时需要重复读取

同一个 `read_page` 请求，在点击前后会取得不同正文。因此，不能把所有“工具名和参数相同”的请求都当成无用重复。

本例点击工具只返回操作状态，随后还需要读取正文。动态环境中应考虑页面状态变化，再决定缓存或去重策略。


## 5. 浏览器的操作范围

本地课堂实验使用新的浏览器上下文，只访问本地页面，并拦截 HTTP/HTTPS 请求。模型分支也只开放读取和一个指定按钮。

公开页实验另设 URL、主机和方法限制，不提供账号登录、任意点击、上传或提交功能。

网页正文可以提供资料，但不能授权额外操作。用户允许读网页，也不代表允许把其他文件上传给网页。


## 6. 应该保存什么

保存动作日志、前后正文、DOM 和截图，并用同一个观察名称关联。复盘时确认：

- 点击前证据是否隐藏。
- 哪个按钮实际被点击。
- 弹窗怎样处理。
- 何时确认正文可见。
- 最终引用是否来自点击后的正文。

若浏览器或页面失败，保留失败原因。仅启动 Chromium 成功，不能说明网页任务完成。


## 7. 练习与代码说明

尝试把按钮名称改错，观察是否明确失败。再去掉点击后的第二次读取，说明为什么不能确认模型已经看到新正文。

`write_fixture` 生成本地页面，`observe` 保存观察，`run_local_workflow` 执行固定点击和恢复。默认代码会实际启动本地 Chromium，因此需要 Playwright 和已安装的浏览器。

`run_local_agent` 是另一个由模型提出动作的分支，`RUN_LIVE=False` 时不运行。固定脚本通过不等于模型具备同样的恢复能力。

下一页：[公开网页读取与摘要](02-public-page-agent.ipynb)。


In [ ]:
from miniagent_course import ROOT, OUTPUT, load_lesson


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from pathlib import Path
import json
import asyncio
import os


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


def in_browser_thread(function, *args, **kwargs):
    # Notebook 已有事件循环；浏览器从创建到关闭均在这个独立线程内完成。
    def worker():
        previous_policy = asyncio.get_event_loop_policy()
        if os.name == "nt":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        try:
            return function(*args, **kwargs)
        finally:
            asyncio.set_event_loop_policy(previous_policy)
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(worker).result()

In [ ]:
FIXTURE_HTML = """
<!doctype html>
<html lang="zh-CN"><meta charset="utf-8"><title>Agent 浏览器课堂</title>
<style>body{font:20px system-ui;max-width:780px;margin:60px auto;padding:20px;background:#f6f8fc;color:#18263d}button{padding:12px 24px;font:inherit}article{background:white;padding:24px;margin-top:24px}</style>
<h1>Agent 浏览器课堂</h1>
<p>先观察页面，再点击按钮，最后提取证据。</p>
<button id="show">显示证据</button>
<article id="evidence" hidden>Agent 根据工具反馈选择下一步行动。</article>
<script>document.querySelector('#show').onclick=()=>{alert('这是演示提示');setTimeout(()=>{document.querySelector('#evidence').hidden=false},100)};</script>
</html>
"""



def write_fixture(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / "fixture.html"
    path.write_text(FIXTURE_HTML, encoding="utf-8")
    return path

In [ ]:
CHAPTER_OUTPUT = OUTPUT / "06-browser-agent"
fixture_path = write_fixture(CHAPTER_OUTPUT)
print(FIXTURE_HTML)
print("已生成：", fixture_path)

In [ ]:
@contextmanager
def classroom_browser():
    from playwright.sync_api import sync_playwright
    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=True)
        context = browser.new_context(viewport={"width": 1100, "height": 720})
        context.route("http://**/*", lambda route: route.abort())
        context.route("https://**/*", lambda route: route.abort())
        page = context.new_page()
        page.set_default_timeout(2500)
        try:
            yield page
        finally:
            context.close()
            browser.close()


def observe(page, directory, name, events):
    text = page.locator("body").inner_text()
    screenshot = Path(directory) / (name + ".png")
    dom = Path(directory) / (name + ".html")
    page.screenshot(path=str(screenshot), full_page=True)
    dom.write_text(page.content(), encoding="utf-8")
    record = {"event": "observe", "name": name, "url": page.url,
              "text": text, "buttons": page.get_by_role("button").all_text_contents(),
              "screenshot": str(screenshot), "dom": str(dom)}
    events.append(record)
    return record

In [ ]:
def run_local_workflow(fixture, directory, expected_label="显示证据"):
    from playwright.sync_api import TimeoutError as PlaywrightTimeout
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    events = []
    outcome = {"status": "failed", "events": events}
    try:
        with classroom_browser() as page:
            page.goto(Path(fixture).resolve().as_uri(), wait_until="domcontentloaded")
            before = observe(page, directory, "before", events)
            assert not page.locator("#evidence").is_visible()
            try:
                page.get_by_role("button", name="旧按钮名称", exact=True).click(timeout=250)
            except PlaywrightTimeout:
                events.append({"event": "locator_timeout", "action": "reobserve"})
                observe(page, directory, "after-timeout", events)

            def dismiss(dialog):
                events.append({"event": "popup_dismissed", "text": dialog.message})
                dialog.dismiss()

            page.on("dialog", dismiss)
            page.get_by_role("button", name=expected_label, exact=True).click()
            events.append({"event": "click", "label": expected_label})
            page.locator("#evidence").wait_for(state="visible")
            after = observe(page, directory, "after", events)
            evidence = page.locator("#evidence").inner_text()
            assert evidence == "Agent 根据工具反馈选择下一步行动。"
            events.append({"event": "click_and_extract", "text": evidence,
                           "observation": "after"})
            outcome.update(status="completed", before=before, after=after, evidence=evidence)
            events.append({"event": "completed"})
    except Exception as error:
        events.append({"event": "failed", "type": type(error).__name__, "detail": str(error)[:1000]})
        outcome["error"] = str(error)
    finally:
        save_json(directory / "actions.json", events)
        save_json(directory / "result.json", outcome)
    return outcome

In [ ]:
local_result = in_browser_thread(run_local_workflow, fixture_path, CHAPTER_OUTPUT / "local")
for event in local_result["events"]:
    print(event["event"], event.get("name", event.get("text", "")))
assert local_result["status"] == "completed", local_result.get("error")
assert local_result["evidence"] not in local_result["before"]["text"]
assert local_result["evidence"] in local_result["after"]["text"]
assert {"locator_timeout", "popup_dismissed", "click_and_extract"} <= {
    event["event"] for event in local_result["events"]}
print("真实浏览器验收通过：隐藏内容在点击后出现，失败定位与弹窗均有记录。")

In [ ]:
from IPython.display import Image, display
print("点击之前：")
display(Image(filename=local_result["before"]["screenshot"]))
print("点击之后：")
display(Image(filename=local_result["after"]["screenshot"]))

In [ ]:
def navigation_failure(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    missing = directory / "page-that-does-not-exist.html"
    if missing.exists():
        raise ValueError("请为导航反例使用没有该文件的输出目录")
    events = []
    with classroom_browser() as page:
        try:
            page.goto(missing.resolve().as_uri(), wait_until="domcontentloaded", timeout=3000)
        except Exception as error:
            events.append({"event": "navigation_failed", "type": type(error).__name__,
                           "detail": str(error)[:1000], "decision": "stop"})
    save_json(directory / "actions.json", events)
    return events

In [ ]:
wrong_locator = in_browser_thread(run_local_workflow, fixture_path,
                                 CHAPTER_OUTPUT / "wrong-locator", expected_label="不存在的新按钮")
assert wrong_locator["status"] == "failed"
assert wrong_locator["events"][-1]["type"] == "TimeoutError"
print("错误定位终止：", wrong_locator["events"][-1]["type"])

navigation_events = in_browser_thread(navigation_failure, CHAPTER_OUTPUT / "navigation-failure")
assert navigation_events and navigation_events[-1]["event"] == "navigation_failed"
print("导航失败已记录：", navigation_events[-1])

In [ ]:
def run_local_agent(fixture, directory, model="qwen3:1.7b"):
    runtime = load_lesson("03-agent-harness/01-harness.ipynb")
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    events = []
    with classroom_browser() as page:
        page.goto(Path(fixture).resolve().as_uri(), wait_until="domcontentloaded")
        observe(page, directory, "before", events)

        def dismiss(dialog):
            events.append({"event": "popup_dismissed", "text": dialog.message})
            dialog.dismiss()

        page.on("dialog", dismiss)

        def read_page():
            return observe(page, directory, "read-" + str(len(events)), events)["text"]

        def click_evidence(label):
            if label != "显示证据":
                raise ValueError("只能点击课堂页面的证据按钮")
            page.get_by_role("button", name=label, exact=True).click()
            page.locator("#evidence").wait_for(state="visible")
            events.append({"event": "click", "label": label})
            return "证据已显示；请重新读取正文"

        tools = {
            "read_page": runtime.Tool(runtime.schema("read_page", "读取当前可见正文", {}),
                                      read_page, repeatable=True),
            "click_evidence": runtime.Tool(runtime.schema("click_evidence", "点击证据按钮", {
                "label": {"type": "string", "enum": ["显示证据"]}}), click_evidence),
        }
        provider = runtime.OllamaModel(model)
        result = runtime.Harness(provider, tools, max_steps=6, max_tools=5, timeout=90).run(
            "先读取当前课堂页面，再点击显示证据，重新读取出现的正文，最后准确写出那条原文。")
        observe(page, directory, "after", events)
        result["evidence_visible"] = page.locator("#evidence").is_visible()
        result["answer_contains_evidence"] = "Agent 根据工具反馈选择下一步行动。" in result["answer"]
        save_json(directory / "actions.json", events)
        save_json(directory / "model-requests.json", provider.requests)
        save_json(directory / "agent-trace.json", result)
        return result

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    live_result = in_browser_thread(run_local_agent, fixture_path, CHAPTER_OUTPUT / "local-agent")
    print(json.dumps(live_result, ensure_ascii=False, indent=2))
    assert live_result["status"] == "completed"
    assert live_result["evidence_visible"] and live_result["answer_contains_evidence"]
else:
    print("真实模型控制浏览器未执行；已完成的是本地 Chromium 固定工作流与故障实验。")